# 01 - Weaviate Collection Setup

- Vectorizer: `text2vec-huggingface` (`sentence-transformers/all-MiniLM-L6-v2`)
- Reranker: `reranker-cohere` (`rerank-english-v3.0`)

> Change `EMBED_MODEL` to any model from your cluster.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

HF_API_KEY     = os.getenv('HF_API_KEY')
COHERE_API_KEY = os.getenv('COHERE_API_KEY', '')
WEAVIATE_URL   = os.getenv('WEAVIATE_URL', '').rstrip('/')
WEAVIATE_API_KEY = os.getenv('WEAVIATE_API_KEY')
COLLECTION     = 'ScmChunks'
EMBED_MODEL    = 'sentence-transformers/all-MiniLM-L6-v2'
RERANKER_MODEL = 'rerank-english-v3.0'

print(f'Cluster    : {WEAVIATE_URL}')
print(f'Collection : {COLLECTION}')
print(f'Embed model: {EMBED_MODEL}')
print(f'Cohere key : {bool(COHERE_API_KEY)}')

Cluster    : https://cbjb4z7tlcnh0bivfdhg.c0.asia-southeast1.gcp.weaviate.cloud
Collection : ScmChunks
Embed model: sentence-transformers/all-MiniLM-L6-v2
Cohere key : True


## 1. Connect

In [2]:
import weaviate
from weaviate.classes.init import Auth, AdditionalConfig, Timeout

_headers = {}
if HF_API_KEY:
    _headers["X-HuggingFace-Api-Key"] = HF_API_KEY.strip()
if COHERE_API_KEY:
    _headers["X-Cohere-Api-Key"] = COHERE_API_KEY.strip()

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY),
    headers=_headers,
    additional_config=AdditionalConfig(
        timeout=Timeout(init=60, query=30, insert=60)
    ),
    skip_init_checks=True,
)
print(f"Ready: {client.is_ready()}")
existing = list(client.collections.list_all().keys())
print(f'Existing: {existing}')

Ready: True
Existing: ['ScmChunks']


## 2. Drop old collection

In [3]:
if COLLECTION in existing:
    client.collections.delete(COLLECTION)
    print(f'Deleted: {COLLECTION}')
else:
    print('Nothing to delete')

Deleted: ScmChunks


## 3. Create collection

In [4]:
from weaviate.classes.config import Configure, Property, DataType

client.collections.create(
    name=COLLECTION,
    vectorizer_config=Configure.Vectorizer.text2vec_huggingface(model=EMBED_MODEL),
    reranker_config=Configure.Reranker.cohere(model=RERANKER_MODEL),
    properties=[
        Property(name='chunk_id',    data_type=DataType.TEXT),
        Property(name='source_file', data_type=DataType.TEXT),
        Property(name='chunk_idx',   data_type=DataType.INT),
        Property(name='text',        data_type=DataType.TEXT),
        Property(name='char_count',  data_type=DataType.INT),
    ]
)
print(f'Created: {COLLECTION}')

c:\Users\abhir\Downloads\graph-rag-local\.venv\Lib\site-packages\weaviate\warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(


Created: ScmChunks


## 4. Test vectorizer & reranker

In [5]:
from weaviate.classes.query import Rerank

collection = client.collections.get(COLLECTION)

id1 = collection.data.insert({'chunk_id':'t1','source_file':'test.pdf','chunk_idx':0,
                               'text':'Supply chain risk involves identifying disruptions early.','char_count':55})
id2 = collection.data.insert({'chunk_id':'t2','source_file':'test.pdf','chunk_idx':1,
                               'text':'Port congestion delays shipments and raises costs.','char_count':50})

results = collection.query.near_text(
    query='supply chain disruption', limit=5,
    rerank=Rerank(prop='text', query='supply chain disruption'),
    return_properties=['chunk_id','text'],
)
print('Reranked results:')
for i, o in enumerate(results.objects, 1):
    score = o.metadata.rerank_score if o.metadata else None
    print(f'  [{i}] score={score:.4f}  {o.properties["text"][:60]}')

collection.data.delete_by_id(id1)
collection.data.delete_by_id(id2)
client.close()
print('Collection ready')

Reranked results:
  [1] score=0.9830  Supply chain risk involves identifying disruptions early.
  [2] score=0.0008  Port congestion delays shipments and raises costs.
Collection ready
